# Google Play Phase 2 Cadence Test — Run B First Collection

This notebook begins the second controlled cadence test requested after Run A.

The purpose of this first collection is to establish a fresh and clearly recorded starting point for Run B. A follow-up collection will be completed approximately 8–12 hours later using the updated database from this run.

The two Run B collections will use the same controlled setup:

- the same 10 Google Play apps
- the same target of 1,200 newest reviews per app
- the same English and United States settings
- the same Phase 2 SQLite database
- the same database continuation logic
- the same duplicate-prevention logic
- the same run-level and app-level summary structure

The final cadence analysis will distinguish between:

- review IDs that are new to the database
- reviews actually posted between the two Run B collections
- older reviews that appear in the returned review window later
- reviews with missing or unusable timestamps

No cadence recommendation will be made from this first collection alone. The recommendation will be based on the completed Run B comparison and the earlier controlled results.

In [1]:
!pip -q install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00


## 1. Import packages and set the fixed Run B configuration

This first collection continues from the database produced after Cadence Run A.

The collection settings remain unchanged so that the results are comparable with the earlier controlled runs.

In [2]:
import os
import re
import json
import time
import shutil
import sqlite3
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from google_play_scraper import reviews, Sort

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATABASE_DIR = BASE_DIR / "database"
OUTPUT_DIR = BASE_DIR / "outputs"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / "google_play_reviews.sqlite"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
SORT_METHOD = Sort.NEWEST

TARGET_REVIEWS_PER_APP = 1200
REQUEST_SLEEP_SECONDS = 2

RUN_LABEL = "phase2_cadence_runB_first_collection"
FREQUENCY_LABEL = "runB_first_collection"

RUN_ID = (
    f"{RUN_LABEL}_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)

print("Run ID:", RUN_ID)
print("Run label:", RUN_LABEL)
print("Frequency label:", FREQUENCY_LABEL)
print("Source:", SOURCE)
print("Language / country:", LANGUAGE, "/", COUNTRY)
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Run ID: phase2_cadence_runB_first_collection_20260714_014620
Run label: phase2_cadence_runB_first_collection
Frequency label: runB_first_collection
Source: google_play
Language / country: en / us
Target reviews per app: 1200
Database path: /content/database/google_play_reviews.sqlite
Output folder: /content/outputs


## 2. Load the database produced after Cadence Run A

This Run B collection must continue from the existing Phase 2 database rather than starting from an empty database.

The uploaded Run A package should contain the database after the completed Day 1, Day 2, Day 3, and Cadence Run A collections.

Before continuing, the database will be checked for:

- the six required Phase 2 tables
- the same 10-app configuration
- the same 1,200-review target
- four completed prior Phase 2 runs
- the completed Cadence Run A record
- matching raw and cleaned review counts

The database will only be copied into the working directory if all checks pass.

In [3]:
from google.colab import files

UPLOAD_DIR = Path("/content/uploaded_runA_files")
EXTRACT_DIR = UPLOAD_DIR / "extracted"
NESTED_DB_DIR = UPLOAD_DIR / "database_extracted"

# Remove files from an earlier failed attempt.
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
NESTED_DB_DIR.mkdir(parents=True, exist_ok=True)

print("Upload this file:")
print("phase2_cadence_runA_github_upload_files_20260709_211803_utc.zip")

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one Run A GitHub zip file.")

uploaded_name = next(iter(uploaded))
uploaded_path = Path("/content") / uploaded_name
saved_upload_path = UPLOAD_DIR / uploaded_name

shutil.move(str(uploaded_path), str(saved_upload_path))

if not zipfile.is_zipfile(saved_upload_path):
    raise ValueError("The uploaded file is not a valid zip file.")

# Extract the Run A GitHub package.
with zipfile.ZipFile(saved_upload_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Outer zip extracted successfully.")

# Locate the compressed Run A SQLite database.
nested_db_zips = list(
    EXTRACT_DIR.rglob("google_play_reviews_after_cadence_runA.sqlite.zip")
)

if len(nested_db_zips) != 1:
    raise FileNotFoundError(
        "Expected exactly one compressed Run A SQLite database, "
        f"but found {len(nested_db_zips)}."
    )

nested_db_zip = nested_db_zips[0]

if not zipfile.is_zipfile(nested_db_zip):
    raise ValueError("The compressed Run A database is not a valid zip file.")

with zipfile.ZipFile(nested_db_zip, "r") as zf:
    zf.extractall(NESTED_DB_DIR)

db_candidates = [
    p for p in NESTED_DB_DIR.rglob("google_play_reviews.sqlite")
    if p.is_file()
]

if len(db_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one google_play_reviews.sqlite file, "
        f"but found {len(db_candidates)}."
    )

source_db_path = db_candidates[0]

required_tables = [
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
]

with sqlite3.connect(source_db_path) as test_conn:
    existing_tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """,
        test_conn,
    )["name"].tolist()

    missing_tables = [
        table_name
        for table_name in required_tables
        if table_name not in existing_tables
    ]

    if missing_tables:
        raise ValueError(
            f"Uploaded database is missing required tables: {missing_tables}"
        )

    raw_rows = int(
        pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
            test_conn,
        )["n"].iloc[0]
    )

    cleaned_rows = int(
        pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
            test_conn,
        )["n"].iloc[0]
    )

    app_count = int(
        pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM phase2_apps",
            test_conn,
        )["n"].iloc[0]
    )

    prior_runs_df = pd.read_sql_query(
        """
        SELECT
            run_id,
            run_label,
            frequency_label,
            target_reviews_per_app,
            app_count,
            run_started_at,
            run_finished_at,
            status
        FROM phase2_ingestion_runs
        ORDER BY run_started_at
        """,
        test_conn,
    )

expected_runA_label = "phase2_cadence_runA_twice_daily_test"

validation_checks = {
    "raw_and_cleaned_counts_match": raw_rows == cleaned_rows,
    "expected_10_apps": app_count == 10,
    "expected_4_prior_runs": len(prior_runs_df) == 4,
    "all_prior_runs_completed": prior_runs_df["status"].eq("completed").all(),
    "same_1200_review_target": prior_runs_df[
        "target_reviews_per_app"
    ].eq(1200).all(),
    "same_10_app_run_setting": prior_runs_df["app_count"].eq(10).all(),
    "cadence_runA_present": prior_runs_df[
        "run_label"
    ].eq(expected_runA_label).any(),
}

failed_checks = [
    check_name
    for check_name, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        f"Run A database validation failed: {failed_checks}"
    )

# Copy the validated Run A database into the working database folder.
if DB_PATH.exists():
    DB_PATH.unlink()

shutil.copy2(source_db_path, DB_PATH)

print()
print("Run A database validated and copied successfully.")
print("Database path:", DB_PATH)
print("Database size:", round(DB_PATH.stat().st_size / (1024 ** 2), 2), "MB")
print("Raw review rows:", raw_rows)
print("Cleaned review rows:", cleaned_rows)
print("Apps:", app_count)
print("Completed prior Phase 2 runs:", len(prior_runs_df))

display(prior_runs_df)

Upload this file:
phase2_cadence_runA_github_upload_files_20260709_211803_utc.zip


Saving phase2_cadence_runA_github_upload_files_20260709_211803_utc.zip to phase2_cadence_runA_github_upload_files_20260709_211803_utc.zip
Outer zip extracted successfully.

Run A database validated and copied successfully.
Database path: /content/database/google_play_reviews.sqlite
Database size: 55.52 MB
Raw review rows: 22210
Cleaned review rows: 22210
Apps: 10
Completed prior Phase 2 runs: 4


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,status
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,1200,10,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,completed
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,1200,10,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,completed
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,1200,10,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,completed
3,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,1200,10,2026-07-09T21:14:19.095514+00:00,2026-07-09T21:15:07.605058+00:00,completed


## 3. Verify the fixed app list and record the pre-run database snapshot

Before starting the Run B first collection, this section confirms that the controlled setup still matches the earlier Phase 2 runs.

The checks cover:

- the exact same 10 apps
- the same Google Play, English, and United States settings
- the same 1,200-review target per app
- the completed Run A database
- matching raw and cleaned review counts
- the latest completed run timestamp
- the current database size and app-level record counts

This snapshot will be used to measure database growth and identify records inserted during this collection.

In [4]:
# Close an older connection only if this cell is rerun.
if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

required_tables = [
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
]

existing_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """,
    conn,
)["name"].tolist()

missing_tables = [
    table_name
    for table_name in required_tables
    if table_name not in existing_tables
]

if missing_tables:
    raise ValueError(
        f"Database is missing required Phase 2 tables: {missing_tables}"
    )

# Read the fixed app configuration directly from the existing database.
app_config_df = pd.read_sql_query(
    """
    SELECT
        app_name,
        app_id,
        source,
        language,
        country,
        title_from_store,
        score_from_store,
        ratings_from_store,
        installs_from_store
    FROM phase2_apps
    ORDER BY rowid
    """,
    conn,
)

expected_apps = [
    ("YouTube", "com.google.android.youtube"),
    ("TikTok", "com.zhiliaoapp.musically"),
    ("Spotify", "com.spotify.music"),
    ("Instagram", "com.instagram.android"),
    ("Uber", "com.ubercab"),
    ("DoorDash", "com.dd.doordash"),
    ("Duolingo", "com.duolingo"),
    ("Google Maps", "com.google.android.apps.maps"),
    ("Netflix", "com.netflix.mediaclient"),
    ("Reddit", "com.reddit.frontpage"),
]

actual_apps = list(
    app_config_df[["app_name", "app_id"]]
    .itertuples(index=False, name=None)
)

if actual_apps != expected_apps:
    raise ValueError(
        "The app list or app order does not match the fixed "
        "10-app Phase 2 configuration."
    )

if not app_config_df["source"].eq(SOURCE).all():
    raise ValueError("The database source does not match Google Play.")

if not app_config_df["language"].eq(LANGUAGE).all():
    raise ValueError("The database language does not match English.")

if not app_config_df["country"].eq(COUNTRY).all():
    raise ValueError("The database country does not match the United States.")

if TARGET_REVIEWS_PER_APP != 1200:
    raise ValueError("The controlled target must remain 1,200 reviews per app.")

# Read the complete prior run history.
prior_runs_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total
    FROM phase2_ingestion_runs
    ORDER BY run_started_at
    """,
    conn,
)

if len(prior_runs_df) != 4:
    raise ValueError(
        f"Expected exactly 4 completed prior runs, "
        f"but found {len(prior_runs_df)}."
    )

if not prior_runs_df["status"].eq("completed").all():
    raise ValueError("At least one prior Phase 2 run is not completed.")

if not prior_runs_df["target_reviews_per_app"].eq(1200).all():
    raise ValueError("A prior run used a different review target.")

if not prior_runs_df["app_count"].eq(10).all():
    raise ValueError("A prior run used a different app count.")

latest_prior_run = prior_runs_df.iloc[-1]

EXPECTED_PREVIOUS_RUN_LABEL = "phase2_cadence_runA_twice_daily_test"

if latest_prior_run["run_label"] != EXPECTED_PREVIOUS_RUN_LABEL:
    raise ValueError(
        "The latest completed run is not the expected Cadence Run A."
    )

PREVIOUS_RUN_ID = str(latest_prior_run["run_id"])
PREVIOUS_RUN_LABEL = str(latest_prior_run["run_label"])
PREVIOUS_RUN_STARTED_AT = str(latest_prior_run["run_started_at"])
PREVIOUS_RUN_FINISHED_AT = str(latest_prior_run["run_finished_at"])

# Record the database state before Run B first collection.
db_size_before_mb = float(
    DB_PATH.stat().st_size / (1024 ** 2)
)

raw_rows_before = int(
    pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM phase2_reviews_raw",
        conn,
    )["n"].iloc[0]
)

cleaned_rows_before = int(
    pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM phase2_reviews_cleaned",
        conn,
    )["n"].iloc[0]
)

quality_flag_rows_before = int(
    pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM phase2_quality_flags",
        conn,
    )["n"].iloc[0]
)

app_summary_rows_before = int(
    pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM phase2_app_run_summary",
        conn,
    )["n"].iloc[0]
)

if raw_rows_before != cleaned_rows_before:
    raise ValueError(
        "Raw and cleaned review counts do not match before Run B."
    )

if raw_rows_before != 22210:
    raise ValueError(
        f"Expected 22,210 reviews after Run A, "
        f"but found {raw_rows_before:,}."
    )

pre_run_app_snapshot_df = pd.read_sql_query(
    """
    SELECT
        a.app_name,
        a.app_id,
        COUNT(r.review_key) AS database_review_rows,
        MIN(r.review_created_at) AS earliest_review_timestamp,
        MAX(r.review_created_at) AS latest_review_timestamp
    FROM phase2_apps AS a
    LEFT JOIN phase2_reviews_raw AS r
        ON a.app_id = r.app_id
    GROUP BY
        a.app_name,
        a.app_id
    ORDER BY a.rowid
    """,
    conn,
)

snapshot_at = datetime.now(timezone.utc).isoformat()

pre_run_snapshot_df = pd.DataFrame(
    [
        {
            "snapshot_at": snapshot_at,
            "database_path": str(DB_PATH),
            "database_size_mb": db_size_before_mb,
            "raw_review_rows": raw_rows_before,
            "cleaned_review_rows": cleaned_rows_before,
            "quality_flag_rows": quality_flag_rows_before,
            "app_run_summary_rows": app_summary_rows_before,
            "completed_prior_runs": len(prior_runs_df),
            "previous_run_id": PREVIOUS_RUN_ID,
            "previous_run_label": PREVIOUS_RUN_LABEL,
            "previous_run_started_at": PREVIOUS_RUN_STARTED_AT,
            "previous_run_finished_at": PREVIOUS_RUN_FINISHED_AT,
        }
    ]
)

print("Pre-run database validation passed.")
print("-" * 70)
print("Fixed apps:", len(app_config_df))
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)
print("Raw review rows before Run B:", f"{raw_rows_before:,}")
print("Cleaned review rows before Run B:", f"{cleaned_rows_before:,}")
print("Database size before Run B:", f"{db_size_before_mb:.2f} MB")
print("Completed prior runs:", len(prior_runs_df))
print("Previous completed run:", PREVIOUS_RUN_LABEL)
print("Previous run finished at:", PREVIOUS_RUN_FINISHED_AT)
print("Snapshot recorded at:", snapshot_at)

print("\nFixed app configuration:")
display(
    app_config_df[
        [
            "app_name",
            "app_id",
            "source",
            "language",
            "country",
        ]
    ]
)

print("\nPre-run app-level database snapshot:")
display(pre_run_app_snapshot_df)

print("\nRun history through Cadence Run A:")
display(prior_runs_df)

Pre-run database validation passed.
----------------------------------------------------------------------
Fixed apps: 10
Target reviews per app: 1200
Raw review rows before Run B: 22,210
Cleaned review rows before Run B: 22,210
Database size before Run B: 55.52 MB
Completed prior runs: 4
Previous completed run: phase2_cadence_runA_twice_daily_test
Previous run finished at: 2026-07-09T21:15:07.605058+00:00
Snapshot recorded at: 2026-07-14T01:51:46.769761+00:00

Fixed app configuration:


,app_name,app_id,source,language,country
0,YouTube,com.google.android.youtube,google_play,en,us
1,TikTok,com.zhiliaoapp.musically,google_play,en,us
2,Spotify,com.spotify.music,google_play,en,us
3,Instagram,com.instagram.android,google_play,en,us
4,Uber,com.ubercab,google_play,en,us
5,DoorDash,com.dd.doordash,google_play,en,us
6,Duolingo,com.duolingo,google_play,en,us
7,Google Maps,com.google.android.apps.maps,google_play,en,us
8,Netflix,com.netflix.mediaclient,google_play,en,us
9,Reddit,com.reddit.frontpage,google_play,en,us



Pre-run app-level database snapshot:


,app_name,app_id,database_review_rows,earliest_review_timestamp,latest_review_timestamp
0,YouTube,com.google.android.youtube,3632,2026-07-06T09:09:19+00:00,2026-07-08T21:13:22+00:00
1,TikTok,com.zhiliaoapp.musically,2465,2026-07-05T03:11:35+00:00,2026-07-08T21:14:15+00:00
2,Spotify,com.spotify.music,2518,2026-07-05T12:54:10+00:00,2026-07-08T21:14:19+00:00
3,Instagram,com.instagram.android,3650,2026-07-06T13:35:02+00:00,2026-07-08T21:12:32+00:00
4,Uber,com.ubercab,1932,2026-07-04T03:52:16+00:00,2026-07-08T21:12:10+00:00
5,DoorDash,com.dd.doordash,1403,2026-06-28T23:28:16+00:00,2026-07-08T21:03:08+00:00
6,Duolingo,com.duolingo,2090,2026-07-06T04:53:03+00:00,2026-07-08T15:57:17+00:00
7,Google Maps,com.google.android.apps.maps,1618,2026-06-30T02:30:54+00:00,2026-07-08T21:14:19+00:00
8,Netflix,com.netflix.mediaclient,1471,2026-06-27T05:44:07+00:00,2026-07-08T21:01:32+00:00
9,Reddit,com.reddit.frontpage,1431,2026-06-27T04:42:08+00:00,2026-07-08T20:43:50+00:00



Run history through Cadence Run A:


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb,errors_total
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,1200,10,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.271117,completed,12000,12000,0,0,12000,12000,1.832031,25.492188,23.660156,0
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,1200,10,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.799585,completed,12000,156,11844,12000,12156,156,25.492188,30.187500,4.695312,0
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,1200,10,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,29.269241,completed,12000,5659,6341,12156,17815,5659,30.187500,43.761719,13.574219,0
3,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,1200,10,2026-07-09T21:14:19.095514+00:00,2026-07-09T21:15:07.605058+00:00,28.142863,completed,12000,4395,7605,17815,22210,4395,43.761719,55.523438,11.761719,0


## 4. Define the normalization and database insertion functions

These functions preserve the same Phase 2 processing and duplicate-prevention logic used in the earlier controlled runs.

Each valid review is identified by a deterministic key created from:

`source + app_id + review_id`

The database uses this key to prevent the same review from being inserted more than once.

For each newly inserted raw review, the notebook also creates:

- a cleaned review record
- applicable data-quality flags
- the original source response stored as JSON

The schema and review identity logic are not changed for Run B.

In [5]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def to_iso_utc(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, pd.Timestamp):
        dt = value.to_pydatetime()
    elif isinstance(value, datetime):
        dt = value
    else:
        try:
            dt = pd.to_datetime(value).to_pydatetime()
        except Exception:
            return str(value)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc).isoformat(timespec="seconds")


def make_hash(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()


def make_review_key(source, app_id, review_id):
    return make_hash(
        f"{source}|{app_id}|{review_id}"
    )


def make_flag_id(run_id, review_key, flag_name):
    return make_hash(
        f"{run_id}|{review_key}|{flag_name}"
    )


def clean_content(text):
    if text is None:
        return None

    cleaned = str(text).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned


def json_ready(value):
    if isinstance(value, (datetime, pd.Timestamp)):
        return to_iso_utc(value)

    return value


def raw_review_to_json(raw_review):
    cleaned = {}

    for key, value in raw_review.items():
        cleaned[key] = json_ready(value)

    return json.dumps(
        cleaned,
        ensure_ascii=False,
    )


def normalize_review(
    raw_review,
    app_id,
    app_name,
    fetched_at,
    run_id,
):
    review_id = raw_review.get("reviewId")

    review_key = None

    if review_id:
        review_key = make_review_key(
            SOURCE,
            app_id,
            review_id,
        )

    content_raw = raw_review.get("content")
    reply_content_raw = raw_review.get("replyContent")

    app_version = (
        raw_review.get("reviewCreatedVersion")
        or raw_review.get("appVersion")
    )

    row = {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": raw_review.get("userName"),
        "user_image": raw_review.get("userImage"),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get("thumbsUpCount"),
        "review_created_at": to_iso_utc(
            raw_review.get("at")
        ),
        "reply_content_raw": reply_content_raw,
        "replied_at": to_iso_utc(
            raw_review.get("repliedAt")
        ),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": run_id,
        "raw_json": raw_review_to_json(raw_review),
    }

    return row


def make_cleaned_row(raw_row, cleaned_at):
    content_cleaned = clean_content(
        raw_row.get("content_raw")
    )

    return {
        "review_key": raw_row.get("review_key"),
        "source": raw_row.get("source"),
        "app_id": raw_row.get("app_id"),
        "content_cleaned": content_cleaned,
        "content_length": (
            len(content_cleaned)
            if content_cleaned is not None
            else None
        ),
        "has_developer_reply": (
            1
            if raw_row.get("reply_content_raw")
            not in [None, ""]
            else 0
        ),
        "score": raw_row.get("score"),
        "review_created_at": raw_row.get(
            "review_created_at"
        ),
        "app_version": raw_row.get("app_version"),
        "cleaned_at": cleaned_at,
        "run_id": raw_row.get("run_id"),
    }


def generate_quality_flags(raw_row, run_id):
    flags = []
    created_at = utc_now_iso()

    review_key = raw_row.get("review_key")
    app_id = raw_row.get("app_id")

    if review_key is None:
        return flags

    def add_flag(
        flag_name,
        severity,
        flag_value,
    ):
        flags.append(
            {
                "flag_id": make_flag_id(
                    run_id,
                    review_key,
                    flag_name,
                ),
                "review_key": review_key,
                "run_id": run_id,
                "app_id": app_id,
                "flag_name": flag_name,
                "flag_severity": severity,
                "flag_value": flag_value,
                "created_at": created_at,
            }
        )

    if raw_row.get("content_raw") is None:
        add_flag(
            "missing_content",
            "warning",
            "missing",
        )
    elif str(
        raw_row.get("content_raw")
    ).strip() == "":
        add_flag(
            "empty_content",
            "warning",
            "empty",
        )

    if raw_row.get("score") is None:
        add_flag(
            "missing_score",
            "warning",
            "missing",
        )
    elif raw_row.get("score") not in [
        1,
        2,
        3,
        4,
        5,
    ]:
        add_flag(
            "invalid_score",
            "warning",
            str(raw_row.get("score")),
        )

    if raw_row.get("review_created_at") is None:
        add_flag(
            "missing_review_date",
            "warning",
            "missing",
        )

    if raw_row.get("app_version") in [None, ""]:
        add_flag(
            "missing_app_version",
            "info",
            "missing",
        )

    if raw_row.get("reply_content_raw") in [
        None,
        "",
    ]:
        add_flag(
            "missing_developer_reply",
            "info",
            "missing",
        )

    return flags


def insert_raw_review(conn, raw_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "app_name",
        "review_id",
        "user_name",
        "user_image",
        "content_raw",
        "score",
        "thumbs_up_count",
        "review_created_at",
        "reply_content_raw",
        "replied_at",
        "app_version",
        "fetched_at",
        "run_id",
        "raw_json",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_raw
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        raw_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


def insert_cleaned_review(conn, cleaned_row):
    columns = [
        "review_key",
        "source",
        "app_id",
        "content_cleaned",
        "content_length",
        "has_developer_reply",
        "score",
        "review_created_at",
        "app_version",
        "cleaned_at",
        "run_id",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_cleaned
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        cleaned_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


def insert_quality_flag(conn, flag_row):
    columns = [
        "flag_id",
        "review_key",
        "run_id",
        "app_id",
        "flag_name",
        "flag_severity",
        "flag_value",
        "created_at",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_quality_flags
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        flag_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


print("Helper functions loaded successfully.")
print(
    "Duplicate identity: "
    "source + app_id + review_id"
)

Helper functions loaded successfully.
Duplicate identity: source + app_id + review_id


## 5. Create the Run B first-collection record

This step creates the database record for the first collection in the second controlled cadence test.

This collection is not treated as evidence for a twice-daily recommendation by itself. Its purpose is to establish a new, precisely recorded starting point for the Run B follow-up collection.

The record keeps the same:

- Phase 2 database
- 10-app configuration
- 1,200-review target per app
- Google Play source settings
- run-tracking structure

In [6]:
existing_run_id_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_ingestion_runs
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

if existing_run_id_count != 0:
    raise ValueError(
        "This Run ID already exists in the database. "
        "Do not insert the same run twice."
    )

run_started_at = utc_now_iso()

apps_included = ", ".join(
    app_config_df["app_name"].tolist()
)

cur.execute(
    """
    INSERT INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
    )
    """,
    (
        str(RUN_ID),
        str(RUN_LABEL),
        "phase2",
        str(FREQUENCY_LABEL),
        str(SOURCE),
        str(LANGUAGE),
        str(COUNTRY),
        int(TARGET_REVIEWS_PER_APP),
        int(len(app_config_df)),
        str(apps_included),
        str(run_started_at),
        "running",
        0,
        0,
        0,
        0,
        0,
        0,
        float(db_size_before_mb),
        int(raw_rows_before),
        (
            "Run B first collection for the second "
            "controlled cadence test. Uses the same "
            "10 apps, the same 1,200-review target, "
            "the existing Run A database, and the "
            "same duplicate-prevention logic. This "
            "collection establishes the timestamp "
            "and database baseline for the Run B "
            "follow-up collection."
        ),
    ),
)

conn.commit()

created_run_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        status,
        review_rows_before,
        db_size_before_mb,
        notes
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(created_run_df) != 1:
    raise ValueError(
        "The Run B first-collection record "
        "was not created correctly."
    )

if created_run_df["status"].iloc[0] != "running":
    raise ValueError(
        "The new run record does not have "
        "the expected running status."
    )

print(
    "Run B first-collection record "
    "created successfully."
)
print("Run started at:", run_started_at)
print("Review rows before:", f"{raw_rows_before:,}")
print(
    "Database size before:",
    f"{db_size_before_mb:.2f} MB",
)

display(created_run_df)

Run B first-collection record created successfully.
Run started at: 2026-07-14T01:54:19.819646+00:00
Review rows before: 22,210
Database size before: 55.52 MB


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,status,review_rows_before,db_size_before_mb,notes
0,phase2_cadence_runB_first_collection_20260714_...,phase2_cadence_runB_first_collection,runB_first_collection,1200,10,2026-07-14T01:54:19.819646+00:00,running,22210,55.523438,Run B first collection for the second controll...


## 6. Run the first collection for Cadence Test Run B

This step collects the 1,200 newest reviews for each of the same 10 apps.

The collection keeps the same source settings and the same duplicate-prevention logic used in the earlier Phase 2 runs.

For each app, the notebook records:

- reviews returned by the source
- unique and repeated review IDs within the returned batch
- records newly inserted into the database
- records skipped because they already exist
- review timestamp range
- data-quality checks
- runtime and collection errors

A review ID that is new to the database is not automatically treated as a newly posted review. The timestamp analysis will be completed separately after the collection.

In [7]:
# Confirm that this collection has not already started.
current_run_status_df = pd.read_sql_query(
    """
    SELECT status
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(current_run_status_df) != 1:
    raise ValueError(
        "The Run B first-collection record was not found."
    )

if current_run_status_df["status"].iloc[0] != "running":
    raise ValueError(
        "This collection is not in running status."
    )

existing_app_summary_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

existing_run_review_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

existing_run_flag_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

if any(
    value != 0
    for value in [
        existing_app_summary_count,
        existing_run_review_count,
        existing_run_flag_count,
    ]
):
    raise ValueError(
        "Run B first collection already contains results. "
        "Do not run this collection cell twice."
    )

collection_start_time = time.time()

app_summaries = []
quality_flags_created = []
new_review_rows = []
errors = []

print("Starting Run B first collection...")
print("Run ID:", RUN_ID)

for i, app_row in app_config_df.iterrows():
    app_start_time = time.time()

    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    print("\n" + "=" * 90)
    print(
        f"[{i + 1}/{len(app_config_df)}] "
        f"Collecting {app_name} ({app_id})"
    )

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    quality_flag_count = 0
    quality_flags_inserted = 0
    error_message = ""

    missing_review_id_count = 0
    missing_content_count = 0
    empty_content_count = 0
    missing_score_count = 0
    invalid_score_count = 0
    missing_review_date_count = 0
    missing_app_version_count = 0
    missing_developer_reply_count = 0

    min_review_date = None
    max_review_date = None

    app_new_review_rows = []
    app_quality_flags_created = []

    conn.execute("SAVEPOINT app_collection")

    try:
        fetched_at = utc_now_iso()

        fetched_reviews, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=SORT_METHOD,
            count=TARGET_REVIEWS_PER_APP,
        )

        records_fetched = int(len(fetched_reviews))

        normalized_rows = [
            normalize_review(
                raw_review,
                app_id,
                app_name,
                fetched_at,
                RUN_ID,
            )
            for raw_review in fetched_reviews
        ]

        valid_review_keys = [
            row["review_key"]
            for row in normalized_rows
            if row.get("review_key") is not None
        ]

        unique_reviews_in_batch = int(
            len(set(valid_review_keys))
        )

        duplicate_reviews_in_batch = int(
            len(valid_review_keys)
            - unique_reviews_in_batch
        )

        review_dates = [
            row["review_created_at"]
            for row in normalized_rows
            if row.get("review_created_at") is not None
        ]

        if review_dates:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)

        for raw_row in normalized_rows:
            if raw_row.get("review_id") in [None, ""]:
                missing_review_id_count += 1
                continue

            if raw_row.get("content_raw") is None:
                missing_content_count += 1
            elif str(
                raw_row.get("content_raw")
            ).strip() == "":
                empty_content_count += 1

            if raw_row.get("score") is None:
                missing_score_count += 1
            elif raw_row.get("score") not in [
                1,
                2,
                3,
                4,
                5,
            ]:
                invalid_score_count += 1

            if raw_row.get("review_created_at") is None:
                missing_review_date_count += 1

            if raw_row.get("app_version") in [None, ""]:
                missing_app_version_count += 1

            if raw_row.get("reply_content_raw") in [
                None,
                "",
            ]:
                missing_developer_reply_count += 1

            inserted_raw = insert_raw_review(
                conn,
                raw_row,
            )

            if inserted_raw == 1:
                cleaned_row = make_cleaned_row(
                    raw_row,
                    utc_now_iso(),
                )

                inserted_cleaned = insert_cleaned_review(
                    conn,
                    cleaned_row,
                )

                if inserted_cleaned != 1:
                    raise ValueError(
                        "A new raw review did not create "
                        "a matching cleaned review."
                    )

                new_records_inserted += 1
                app_new_review_rows.append(raw_row)

            else:
                duplicates_skipped += 1

            flags = generate_quality_flags(
                raw_row,
                RUN_ID,
            )

            quality_flag_count += len(flags)

            for flag in flags:
                inserted_flag = insert_quality_flag(
                    conn,
                    flag,
                )

                quality_flags_inserted += inserted_flag

                if inserted_flag == 1:
                    app_quality_flags_created.append(flag)

        app_runtime_seconds = float(
            time.time() - app_start_time
        )

        app_summary = {
            "run_id": RUN_ID,
            "app_name": app_name,
            "app_id": app_id,
            "target_reviews": int(
                TARGET_REVIEWS_PER_APP
            ),
            "records_fetched": int(records_fetched),
            "unique_reviews_in_batch": int(
                unique_reviews_in_batch
            ),
            "duplicate_reviews_in_batch": int(
                duplicate_reviews_in_batch
            ),
            "new_records_inserted": int(
                new_records_inserted
            ),
            "duplicates_skipped": int(
                duplicates_skipped
            ),
            "runtime_seconds": app_runtime_seconds,
            "min_review_date": min_review_date,
            "max_review_date": max_review_date,
            "missing_review_id_count": int(
                missing_review_id_count
            ),
            "missing_content_count": int(
                missing_content_count
            ),
            "empty_content_count": int(
                empty_content_count
            ),
            "missing_score_count": int(
                missing_score_count
            ),
            "invalid_score_count": int(
                invalid_score_count
            ),
            "missing_review_date_count": int(
                missing_review_date_count
            ),
            "missing_app_version_count": int(
                missing_app_version_count
            ),
            "missing_developer_reply_count": int(
                missing_developer_reply_count
            ),
            "quality_flag_count": int(
                quality_flag_count
            ),
            "error_message": error_message,
        }

        cur.execute(
            """
            INSERT INTO phase2_app_run_summary (
                run_id,
                app_name,
                app_id,
                target_reviews,
                records_fetched,
                unique_reviews_in_batch,
                duplicate_reviews_in_batch,
                new_records_inserted,
                duplicates_skipped,
                runtime_seconds,
                min_review_date,
                max_review_date,
                missing_review_id_count,
                missing_content_count,
                empty_content_count,
                missing_score_count,
                invalid_score_count,
                missing_review_date_count,
                missing_app_version_count,
                missing_developer_reply_count,
                quality_flag_count,
                error_message
            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            (
                str(app_summary["run_id"]),
                str(app_summary["app_name"]),
                str(app_summary["app_id"]),
                int(app_summary["target_reviews"]),
                int(app_summary["records_fetched"]),
                int(
                    app_summary[
                        "unique_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "duplicate_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "new_records_inserted"
                    ]
                ),
                int(
                    app_summary[
                        "duplicates_skipped"
                    ]
                ),
                float(
                    app_summary["runtime_seconds"]
                ),
                app_summary["min_review_date"],
                app_summary["max_review_date"],
                int(
                    app_summary[
                        "missing_review_id_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "empty_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "invalid_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_review_date_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_app_version_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_developer_reply_count"
                    ]
                ),
                int(
                    app_summary[
                        "quality_flag_count"
                    ]
                ),
                str(app_summary["error_message"]),
            ),
        )

        conn.execute(
            "RELEASE SAVEPOINT app_collection"
        )
        conn.commit()

        app_summaries.append(app_summary)
        new_review_rows.extend(app_new_review_rows)
        quality_flags_created.extend(
            app_quality_flags_created
        )

    except Exception as e:
        conn.execute(
            "ROLLBACK TO SAVEPOINT app_collection"
        )
        conn.execute(
            "RELEASE SAVEPOINT app_collection"
        )
        conn.commit()

        error_message = repr(e)

        errors.append(
            {
                "app_name": app_name,
                "app_id": app_id,
                "error_message": error_message,
            }
        )

        app_runtime_seconds = float(
            time.time() - app_start_time
        )

        app_summary = {
            "run_id": RUN_ID,
            "app_name": app_name,
            "app_id": app_id,
            "target_reviews": int(
                TARGET_REVIEWS_PER_APP
            ),
            "records_fetched": int(records_fetched),
            "unique_reviews_in_batch": int(
                unique_reviews_in_batch
            ),
            "duplicate_reviews_in_batch": int(
                duplicate_reviews_in_batch
            ),
            "new_records_inserted": 0,
            "duplicates_skipped": 0,
            "runtime_seconds": app_runtime_seconds,
            "min_review_date": min_review_date,
            "max_review_date": max_review_date,
            "missing_review_id_count": int(
                missing_review_id_count
            ),
            "missing_content_count": int(
                missing_content_count
            ),
            "empty_content_count": int(
                empty_content_count
            ),
            "missing_score_count": int(
                missing_score_count
            ),
            "invalid_score_count": int(
                invalid_score_count
            ),
            "missing_review_date_count": int(
                missing_review_date_count
            ),
            "missing_app_version_count": int(
                missing_app_version_count
            ),
            "missing_developer_reply_count": int(
                missing_developer_reply_count
            ),
            "quality_flag_count": 0,
            "error_message": error_message,
        }

        app_summaries.append(app_summary)

        cur.execute(
            """
            INSERT INTO phase2_app_run_summary (
                run_id,
                app_name,
                app_id,
                target_reviews,
                records_fetched,
                unique_reviews_in_batch,
                duplicate_reviews_in_batch,
                new_records_inserted,
                duplicates_skipped,
                runtime_seconds,
                min_review_date,
                max_review_date,
                missing_review_id_count,
                missing_content_count,
                empty_content_count,
                missing_score_count,
                invalid_score_count,
                missing_review_date_count,
                missing_app_version_count,
                missing_developer_reply_count,
                quality_flag_count,
                error_message
            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            (
                str(app_summary["run_id"]),
                str(app_summary["app_name"]),
                str(app_summary["app_id"]),
                int(app_summary["target_reviews"]),
                int(app_summary["records_fetched"]),
                int(
                    app_summary[
                        "unique_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "duplicate_reviews_in_batch"
                    ]
                ),
                0,
                0,
                float(
                    app_summary["runtime_seconds"]
                ),
                app_summary["min_review_date"],
                app_summary["max_review_date"],
                int(
                    app_summary[
                        "missing_review_id_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "empty_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "invalid_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_review_date_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_app_version_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_developer_reply_count"
                    ]
                ),
                0,
                str(app_summary["error_message"]),
            ),
        )

        conn.commit()

        print("ERROR:", error_message)

    print(
        f"Fetched={app_summary['records_fetched']:,} | "
        f"New inserts="
        f"{app_summary['new_records_inserted']:,} | "
        f"Duplicates skipped="
        f"{app_summary['duplicates_skipped']:,} | "
        f"Runtime="
        f"{app_summary['runtime_seconds']:.2f}s | "
        f"Quality flags="
        f"{app_summary['quality_flag_count']:,}"
    )

    time.sleep(REQUEST_SLEEP_SECONDS)

collection_runtime_seconds = float(
    time.time() - collection_start_time
)

print("\n" + "=" * 90)
print("Run B first collection finished.")
print(
    f"Collection runtime: "
    f"{collection_runtime_seconds:.2f} seconds"
)
print("Apps processed:", len(app_summaries))
print("Apps with errors:", len(errors))

Starting Run B first collection...
Run ID: phase2_cadence_runB_first_collection_20260714_014620

[1/10] Collecting YouTube (com.google.android.youtube)
Fetched=1,200 | New inserts=1,193 | Duplicates skipped=7 | Runtime=2.15s | Quality flags=1,231

[2/10] Collecting TikTok (com.zhiliaoapp.musically)
Fetched=1,200 | New inserts=1,188 | Duplicates skipped=12 | Runtime=1.33s | Quality flags=627

[3/10] Collecting Spotify (com.spotify.music)
Fetched=1,200 | New inserts=1,200 | Duplicates skipped=0 | Runtime=0.74s | Quality flags=1,261

[4/10] Collecting Instagram (com.instagram.android)
Fetched=1,200 | New inserts=1,199 | Duplicates skipped=1 | Runtime=0.75s | Quality flags=1,584

[5/10] Collecting Uber (com.ubercab)
Fetched=1,200 | New inserts=1,199 | Duplicates skipped=1 | Runtime=0.90s | Quality flags=1,356

[6/10] Collecting DoorDash (com.dd.doordash)
Fetched=1,200 | New inserts=547 | Duplicates skipped=653 | Runtime=0.93s | Quality flags=1,333

[7/10] Collecting Duolingo (com.duolingo)

## 7. Finalize the first collection and validate the updated database

After collection, this section verifies that the run was written to the database correctly.

The checks confirm that:

- all 10 apps were processed
- exactly 12,000 reviews were returned
- new inserts and skipped duplicates reconcile to the fetched total
- every new raw review has one matching cleaned review
- the reported database growth matches the inserted review count
- no duplicate review identities were created
- no orphan cleaned records or quality flags were created
- the run-level database record matches the app-level results

The run is marked as completed only after all required checks pass.

In [8]:
app_summary_df = pd.DataFrame(app_summaries)
error_df = pd.DataFrame(errors)

if len(app_summary_df) != 10:
    raise ValueError(
        f"Expected 10 app summaries, but found {len(app_summary_df)}."
    )

if app_summary_df["app_id"].nunique() != 10:
    raise ValueError(
        "The app-level result does not contain 10 unique apps."
    )

expected_app_ids = app_config_df["app_id"].tolist()
actual_app_ids = app_summary_df["app_id"].tolist()

if actual_app_ids != expected_app_ids:
    raise ValueError(
        "The app order or app IDs changed during collection."
    )

records_fetched_total = int(
    app_summary_df["records_fetched"].sum()
)

new_records_inserted_total = int(
    app_summary_df["new_records_inserted"].sum()
)

duplicates_skipped_total = int(
    app_summary_df["duplicates_skipped"].sum()
)

errors_total = int(len(error_df))

quality_flag_total = int(
    app_summary_df["quality_flag_count"].sum()
)

# Keep runtime comparable with the previous Phase 2 runs by
# summing app-level processing time and excluding sleep intervals.
processing_runtime_seconds = float(
    app_summary_df["runtime_seconds"].sum()
)

if records_fetched_total != 12000:
    raise ValueError(
        f"Expected 12,000 fetched reviews, "
        f"but found {records_fetched_total:,}."
    )

if (
    new_records_inserted_total
    + duplicates_skipped_total
    != records_fetched_total
):
    raise ValueError(
        "New inserts plus skipped duplicates do not "
        "match the fetched total."
    )

if errors_total != 0:
    raise ValueError(
        f"The collection contains {errors_total} app error(s)."
    )

if not app_summary_df["records_fetched"].eq(1200).all():
    raise ValueError(
        "At least one app did not return exactly 1,200 reviews."
    )

app_summary_rows_in_db = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

run_raw_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

run_cleaned_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

quality_flags_inserted = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

raw_rows_after = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        """,
        conn,
    )["n"].iloc[0]
)

cleaned_rows_after = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        """,
        conn,
    )["n"].iloc[0]
)

review_rows_growth = int(
    raw_rows_after - raw_rows_before
)

db_size_after_mb = float(
    DB_PATH.stat().st_size / (1024 ** 2)
)

db_size_growth_mb = float(
    db_size_after_mb - db_size_before_mb
)

duplicate_identity_groups = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM (
            SELECT
                source,
                app_id,
                review_id,
                COUNT(*) AS row_count
            FROM phase2_reviews_raw
            GROUP BY
                source,
                app_id,
                review_id
            HAVING COUNT(*) > 1
        )
        """,
        conn,
    )["n"].iloc[0]
)

raw_without_cleaned = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw AS r
        LEFT JOIN phase2_reviews_cleaned AS c
            ON r.review_key = c.review_key
        WHERE c.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

cleaned_without_raw = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned AS c
        LEFT JOIN phase2_reviews_raw AS r
            ON c.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

quality_flags_without_raw = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags AS q
        LEFT JOIN phase2_reviews_raw AS r
            ON q.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

validation_checks = {
    "10_app_summary_rows": app_summary_rows_in_db == 10,
    "12000_reviews_fetched": records_fetched_total == 12000,
    "new_plus_duplicates_match_fetched": (
        new_records_inserted_total
        + duplicates_skipped_total
        == records_fetched_total
    ),
    "run_raw_rows_match_new_inserts": (
        run_raw_rows == new_records_inserted_total
    ),
    "run_cleaned_rows_match_new_inserts": (
        run_cleaned_rows == new_records_inserted_total
    ),
    "database_growth_matches_new_inserts": (
        review_rows_growth == new_records_inserted_total
    ),
    "raw_and_cleaned_totals_match": (
        raw_rows_after == cleaned_rows_after
    ),
    "no_duplicate_review_identities": (
        duplicate_identity_groups == 0
    ),
    "no_raw_rows_without_cleaned_rows": (
        raw_without_cleaned == 0
    ),
    "no_cleaned_rows_without_raw_rows": (
        cleaned_without_raw == 0
    ),
    "no_orphan_quality_flags": (
        quality_flags_without_raw == 0
    ),
    "no_app_errors": errors_total == 0,
}

failed_checks = [
    check_name
    for check_name, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        f"Post-run validation failed: {failed_checks}"
    )

run_finished_at = utc_now_iso()

cur.execute(
    """
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?
    """,
    (
        str(run_finished_at),
        float(processing_runtime_seconds),
        "completed",
        int(records_fetched_total),
        int(new_records_inserted_total),
        int(duplicates_skipped_total),
        int(errors_total),
        int(quality_flag_total),
        int(quality_flags_inserted),
        float(db_size_after_mb),
        float(db_size_growth_mb),
        int(raw_rows_after),
        int(review_rows_growth),
        str(RUN_ID),
    ),
)

conn.commit()

completed_run_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(completed_run_df) != 1:
    raise ValueError(
        "The completed run record could not be verified."
    )

if completed_run_df["status"].iloc[0] != "completed":
    raise ValueError(
        "The Run B first collection was not marked completed."
    )

app_summary_df["duplicate_rate"] = (
    app_summary_df["duplicates_skipped"]
    / app_summary_df["records_fetched"]
)

app_summary_df["new_insert_rate"] = (
    app_summary_df["new_records_inserted"]
    / app_summary_df["records_fetched"]
)

app_summary_display_df = app_summary_df[
    [
        "app_name",
        "records_fetched",
        "new_records_inserted",
        "duplicates_skipped",
        "new_insert_rate",
        "duplicate_rate",
        "runtime_seconds",
        "min_review_date",
        "max_review_date",
        "error_message",
    ]
].copy()

app_summary_display_df["new_insert_rate"] = (
    app_summary_display_df["new_insert_rate"]
    .map(lambda value: f"{value:.2%}")
)

app_summary_display_df["duplicate_rate"] = (
    app_summary_display_df["duplicate_rate"]
    .map(lambda value: f"{value:.2%}")
)

validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in validation_checks.items()
    ]
)

print("Run B first collection completed and validated.")
print("-" * 72)
print("Records fetched:", f"{records_fetched_total:,}")
print(
    "New records inserted:",
    f"{new_records_inserted_total:,}",
)
print(
    "Duplicates skipped:",
    f"{duplicates_skipped_total:,}",
)
print(
    "New insert rate:",
    f"{new_records_inserted_total / records_fetched_total:.2%}",
)
print(
    "Duplicate rate:",
    f"{duplicates_skipped_total / records_fetched_total:.2%}",
)
print(
    "Processing runtime:",
    f"{processing_runtime_seconds:.2f} seconds",
)
print("Errors:", errors_total)
print(
    "Review rows before:",
    f"{raw_rows_before:,}",
)
print(
    "Review rows after:",
    f"{raw_rows_after:,}",
)
print(
    "Review row growth:",
    f"{review_rows_growth:,}",
)
print(
    "Database size before:",
    f"{db_size_before_mb:.2f} MB",
)
print(
    "Database size after:",
    f"{db_size_after_mb:.2f} MB",
)
print(
    "Database size growth:",
    f"{db_size_growth_mb:.2f} MB",
)
print(
    "Duplicate identity groups:",
    duplicate_identity_groups,
)
print(
    "Raw rows without cleaned rows:",
    raw_without_cleaned,
)
print(
    "Cleaned rows without raw rows:",
    cleaned_without_raw,
)
print(
    "Orphan quality flags:",
    quality_flags_without_raw,
)

print("\nApp-level collection summary:")
display(app_summary_display_df)

print("\nDatabase validation checks:")
display(validation_df)

print("\nCompleted run record:")
display(completed_run_df)

Run B first collection completed and validated.
------------------------------------------------------------------------
Records fetched: 12,000
New records inserted: 8,638
Duplicates skipped: 3,362
New insert rate: 71.98%
Duplicate rate: 28.02%
Processing runtime: 10.09 seconds
Errors: 0
Review rows before: 22,210
Review rows after: 30,848
Review row growth: 8,638
Database size before: 55.52 MB
Database size after: 74.09 MB
Database size growth: 18.56 MB
Duplicate identity groups: 0
Raw rows without cleaned rows: 0
Cleaned rows without raw rows: 0
Orphan quality flags: 0

App-level collection summary:


,app_name,records_fetched,new_records_inserted,duplicates_skipped,new_insert_rate,duplicate_rate,runtime_seconds,min_review_date,max_review_date,error_message
0,YouTube,1200,1193,7,99.42%,0.58%,2.150620,2026-07-12T07:04:56+00:00,2026-07-13T01:56:32+00:00,
1,TikTok,1200,1188,12,99.00%,1.00%,1.325191,2026-07-11T01:44:22+00:00,2026-07-13T01:54:48+00:00,
2,Spotify,1200,1200,0,100.00%,0.00%,0.744175,2026-07-11T11:16:57+00:00,2026-07-13T01:56:37+00:00,
3,Instagram,1200,1199,1,99.92%,0.08%,0.749634,2026-07-12T12:23:42+00:00,2026-07-13T01:56:17+00:00,
4,Uber,1200,1199,1,99.92%,0.08%,0.902037,2026-07-09T23:42:42+00:00,2026-07-13T01:46:36+00:00,
5,DoorDash,1200,547,653,45.58%,54.42%,0.929154,2026-07-03T19:07:11+00:00,2026-07-13T01:45:26+00:00,
6,Duolingo,1200,5,1195,0.42%,99.58%,0.875325,2026-07-06T18:18:00+00:00,2026-07-12T12:05:22+00:00,
7,Google Maps,1200,866,334,72.17%,27.83%,0.796978,2026-07-07T13:39:04+00:00,2026-07-13T01:48:56+00:00,
8,Netflix,1200,742,458,61.83%,38.17%,0.931145,2026-07-05T21:06:01+00:00,2026-07-13T01:26:29+00:00,
9,Reddit,1200,499,701,41.58%,58.42%,0.689247,2026-07-03T02:21:56+00:00,2026-07-13T01:51:50+00:00,



Database validation checks:


,validation_check,passed
0,10_app_summary_rows,True
1,12000_reviews_fetched,True
2,new_plus_duplicates_match_fetched,True
3,run_raw_rows_match_new_inserts,True
4,run_cleaned_rows_match_new_inserts,True
5,database_growth_matches_new_inserts,True
6,raw_and_cleaned_totals_match,True
7,no_duplicate_review_identities,True
8,no_raw_rows_without_cleaned_rows,True
9,no_cleaned_rows_without_raw_rows,True



Completed run record:


,run_id,run_label,frequency_label,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,quality_flag_total,quality_flags_inserted,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb
0,phase2_cadence_runB_first_collection_20260714_...,phase2_cadence_runB_first_collection,runB_first_collection,2026-07-14T01:54:19.819646+00:00,2026-07-14T01:58:44.989455+00:00,10.093505,completed,12000,8638,3362,0,12630,12630,22210,30848,8638,55.523438,74.085938,18.5625


## 8. Validate whether newly inserted records were actually posted between runs

A review ID can be new to the database even when the review itself was posted earlier.

To separate database novelty from review freshness, each record inserted during this collection is classified using its review timestamp:

- **Posted between runs:** posted after Cadence Run A finished and no later than the time this app was collected
- **Older review surfaced later:** posted on or before Cadence Run A finished but first returned in this collection
- **Timestamp after fetch:** the recorded review timestamp is later than its collection timestamp
- **Missing or unusable timestamp:** the review or collection timestamp cannot be parsed

The comparison uses each record's own `fetched_at` timestamp rather than only the overall run start time. This provides a more accurate upper boundary because the 10 apps were collected sequentially.

The analysis also reports the timestamp range of newly inserted reviews for each app. These results describe the Run B first collection only and do not yet support a final cadence recommendation.

In [9]:
current_run_timing_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        run_started_at,
        run_finished_at,
        status,
        new_records_inserted_total
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

previous_run_timing_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        run_started_at,
        run_finished_at,
        status
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(PREVIOUS_RUN_ID,),
)

if len(current_run_timing_df) != 1:
    raise ValueError(
        "The completed Run B first-collection record was not found."
    )

if len(previous_run_timing_df) != 1:
    raise ValueError(
        "The previous Cadence Run A record was not found."
    )

if current_run_timing_df["status"].iloc[0] != "completed":
    raise ValueError(
        "Run B first collection must be completed before timestamp analysis."
    )

if previous_run_timing_df["status"].iloc[0] != "completed":
    raise ValueError(
        "Cadence Run A is not marked as completed."
    )

previous_run_finished_ts = pd.to_datetime(
    previous_run_timing_df["run_finished_at"].iloc[0],
    utc=True,
    errors="raise",
)

current_run_started_ts = pd.to_datetime(
    current_run_timing_df["run_started_at"].iloc[0],
    utc=True,
    errors="raise",
)

current_run_finished_ts = pd.to_datetime(
    current_run_timing_df["run_finished_at"].iloc[0],
    utc=True,
    errors="raise",
)

gap_before_collection_hours = float(
    (
        current_run_started_ts
        - previous_run_finished_ts
    ).total_seconds()
    / 3600
)

new_review_timestamp_audit_df = pd.read_sql_query(
    """
    SELECT
        review_key,
        app_name,
        app_id,
        review_id,
        review_created_at,
        fetched_at,
        score,
        app_version,
        run_id
    FROM phase2_reviews_raw
    WHERE run_id = ?
    ORDER BY
        app_name,
        review_created_at,
        review_id
    """,
    conn,
    params=(RUN_ID,),
)

expected_new_insert_total = int(
    current_run_timing_df[
        "new_records_inserted_total"
    ].iloc[0]
)

if len(new_review_timestamp_audit_df) != expected_new_insert_total:
    raise ValueError(
        "The timestamp-audit row count does not match "
        "the completed run's new insert count."
    )

new_review_timestamp_audit_df[
    "review_timestamp_parsed"
] = pd.to_datetime(
    new_review_timestamp_audit_df["review_created_at"],
    utc=True,
    errors="coerce",
)

new_review_timestamp_audit_df[
    "fetched_at_parsed"
] = pd.to_datetime(
    new_review_timestamp_audit_df["fetched_at"],
    utc=True,
    errors="coerce",
)


def classify_timestamp(row):
    review_ts = row["review_timestamp_parsed"]
    fetched_ts = row["fetched_at_parsed"]

    if pd.isna(review_ts) or pd.isna(fetched_ts):
        return "missing_or_unusable_timestamp"

    if review_ts <= previous_run_finished_ts:
        return "older_review_surfaced_later"

    if review_ts <= fetched_ts:
        return "posted_between_runs"

    return "timestamp_after_fetch"


new_review_timestamp_audit_df[
    "timestamp_classification"
] = new_review_timestamp_audit_df.apply(
    classify_timestamp,
    axis=1,
)

new_review_timestamp_audit_df[
    "posted_between_runs"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "posted_between_runs"
).astype(int)

new_review_timestamp_audit_df[
    "older_review_surfaced_later"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "older_review_surfaced_later"
).astype(int)

new_review_timestamp_audit_df[
    "timestamp_after_fetch"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "timestamp_after_fetch"
).astype(int)

new_review_timestamp_audit_df[
    "missing_or_unusable_timestamp"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "missing_or_unusable_timestamp"
).astype(int)


def timestamp_to_iso(value):
    if value is None or pd.isna(value):
        return None

    return pd.Timestamp(value).isoformat()


timestamp_summary_rows = []

expected_new_inserts_by_app = (
    app_summary_df
    .set_index("app_id")["new_records_inserted"]
    .to_dict()
)

for _, app_row in app_config_df.iterrows():
    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    app_audit_df = new_review_timestamp_audit_df[
        new_review_timestamp_audit_df["app_id"] == app_id
    ].copy()

    valid_timestamp_df = app_audit_df[
        app_audit_df["review_timestamp_parsed"].notna()
    ]

    posted_between_df = app_audit_df[
        app_audit_df["timestamp_classification"]
        == "posted_between_runs"
    ]

    audited_new_inserts = int(len(app_audit_df))

    posted_between_count = int(
        app_audit_df["posted_between_runs"].sum()
    )

    older_surfaced_count = int(
        app_audit_df[
            "older_review_surfaced_later"
        ].sum()
    )

    timestamp_after_fetch_count = int(
        app_audit_df["timestamp_after_fetch"].sum()
    )

    missing_timestamp_count = int(
        app_audit_df[
            "missing_or_unusable_timestamp"
        ].sum()
    )

    timestamp_summary_rows.append(
        {
            "app_name": app_name,
            "app_id": app_id,
            "expected_new_inserts": int(
                expected_new_inserts_by_app.get(
                    app_id,
                    0,
                )
            ),
            "audited_new_inserts": audited_new_inserts,
            "posted_between_runs_count": (
                posted_between_count
            ),
            "older_reviews_surfaced_later_count": (
                older_surfaced_count
            ),
            "timestamp_after_fetch_count": (
                timestamp_after_fetch_count
            ),
            "missing_or_unusable_timestamp_count": (
                missing_timestamp_count
            ),
            "posted_between_runs_rate": (
                posted_between_count / audited_new_inserts
                if audited_new_inserts > 0
                else 0.0
            ),
            "older_surfaced_rate": (
                older_surfaced_count / audited_new_inserts
                if audited_new_inserts > 0
                else 0.0
            ),
            "new_insert_timestamp_min": (
                timestamp_to_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "new_insert_timestamp_max": (
                timestamp_to_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "posted_between_runs_timestamp_min": (
                timestamp_to_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(posted_between_df) > 0
                else None
            ),
            "posted_between_runs_timestamp_max": (
                timestamp_to_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(posted_between_df) > 0
                else None
            ),
        }
    )

timestamp_summary_df = pd.DataFrame(
    timestamp_summary_rows
)

timestamp_summary_df[
    "classification_total"
] = (
    timestamp_summary_df["posted_between_runs_count"]
    + timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ]
    + timestamp_summary_df[
        "timestamp_after_fetch_count"
    ]
    + timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ]
)

timestamp_validation_checks = {
    "audit_rows_match_run_new_inserts": (
        len(new_review_timestamp_audit_df)
        == expected_new_insert_total
    ),
    "app_level_audit_counts_match_new_inserts": (
        timestamp_summary_df[
            "audited_new_inserts"
        ].equals(
            timestamp_summary_df[
                "expected_new_inserts"
            ]
        )
    ),
    "all_timestamp_classes_reconcile": (
        timestamp_summary_df[
            "classification_total"
        ].equals(
            timestamp_summary_df[
                "audited_new_inserts"
            ]
        )
    ),
    "all_10_apps_in_timestamp_summary": (
        len(timestamp_summary_df) == 10
    ),
    "all_audited_rows_have_current_run_id": (
        new_review_timestamp_audit_df[
            "run_id"
        ].eq(RUN_ID).all()
    ),
}

failed_timestamp_checks = [
    check_name
    for check_name, passed
    in timestamp_validation_checks.items()
    if not passed
]

if failed_timestamp_checks:
    raise ValueError(
        "Timestamp validation failed: "
        f"{failed_timestamp_checks}"
    )

posted_between_runs_total = int(
    timestamp_summary_df[
        "posted_between_runs_count"
    ].sum()
)

older_surfaced_total = int(
    timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ].sum()
)

timestamp_after_fetch_total = int(
    timestamp_summary_df[
        "timestamp_after_fetch_count"
    ].sum()
)

missing_timestamp_total = int(
    timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ].sum()
)

timestamp_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in timestamp_validation_checks.items()
    ]
)

timestamp_summary_display_df = (
    timestamp_summary_df.copy()
)

timestamp_summary_display_df[
    "posted_between_runs_rate"
] = timestamp_summary_display_df[
    "posted_between_runs_rate"
].map(lambda value: f"{value:.2%}")

timestamp_summary_display_df[
    "older_surfaced_rate"
] = timestamp_summary_display_df[
    "older_surfaced_rate"
].map(lambda value: f"{value:.2%}")

print("New-insert timestamp validation passed.")
print("-" * 78)
print(
    "Previous Run A finished at:",
    previous_run_finished_ts.isoformat(),
)
print(
    "Run B first collection started at:",
    current_run_started_ts.isoformat(),
)
print(
    "Gap before this collection:",
    f"{gap_before_collection_hours:.2f} hours",
)
print(
    "New inserts audited:",
    f"{len(new_review_timestamp_audit_df):,}",
)
print(
    "Posted between runs:",
    f"{posted_between_runs_total:,}",
)
print(
    "Older reviews surfaced later:",
    f"{older_surfaced_total:,}",
)
print(
    "Timestamps after fetch:",
    f"{timestamp_after_fetch_total:,}",
)
print(
    "Missing or unusable timestamps:",
    f"{missing_timestamp_total:,}",
)

print("\nApp-level timestamp summary:")
display(
    timestamp_summary_display_df[
        [
            "app_name",
            "audited_new_inserts",
            "posted_between_runs_count",
            "older_reviews_surfaced_later_count",
            "timestamp_after_fetch_count",
            "missing_or_unusable_timestamp_count",
            "posted_between_runs_rate",
            "older_surfaced_rate",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
            "posted_between_runs_timestamp_min",
            "posted_between_runs_timestamp_max",
        ]
    ]
)

print("\nTimestamp validation checks:")
display(timestamp_validation_df)

New-insert timestamp validation passed.
------------------------------------------------------------------------------
Previous Run A finished at: 2026-07-09T21:15:07.605058+00:00
Run B first collection started at: 2026-07-14T01:54:19.819646+00:00
Gap before this collection: 100.65 hours
New inserts audited: 8,638
Posted between runs: 8,015
Older reviews surfaced later: 623
Timestamps after fetch: 0
Missing or unusable timestamps: 0

App-level timestamp summary:


,app_name,audited_new_inserts,posted_between_runs_count,older_reviews_surfaced_later_count,timestamp_after_fetch_count,missing_or_unusable_timestamp_count,posted_between_runs_rate,older_surfaced_rate,new_insert_timestamp_min,new_insert_timestamp_max,posted_between_runs_timestamp_min,posted_between_runs_timestamp_max
0,YouTube,1193,1193,0,0,0,100.00%,0.00%,2026-07-12T07:04:56+00:00,2026-07-13T01:56:32+00:00,2026-07-12T07:04:56+00:00,2026-07-13T01:56:32+00:00
1,TikTok,1188,1188,0,0,0,100.00%,0.00%,2026-07-11T01:44:22+00:00,2026-07-13T01:54:48+00:00,2026-07-11T01:44:22+00:00,2026-07-13T01:54:48+00:00
2,Spotify,1200,1200,0,0,0,100.00%,0.00%,2026-07-11T11:16:57+00:00,2026-07-13T01:56:37+00:00,2026-07-11T11:16:57+00:00,2026-07-13T01:56:37+00:00
3,Instagram,1199,1199,0,0,0,100.00%,0.00%,2026-07-12T12:23:42+00:00,2026-07-13T01:56:17+00:00,2026-07-12T12:23:42+00:00,2026-07-13T01:56:17+00:00
4,Uber,1199,1199,0,0,0,100.00%,0.00%,2026-07-09T23:42:42+00:00,2026-07-13T01:46:36+00:00,2026-07-09T23:42:42+00:00,2026-07-13T01:46:36+00:00
5,DoorDash,547,432,115,0,0,78.98%,21.02%,2026-07-08T21:24:56+00:00,2026-07-13T01:45:26+00:00,2026-07-09T21:27:08+00:00,2026-07-13T01:45:26+00:00
6,Duolingo,5,3,2,0,0,60.00%,40.00%,2026-07-09T05:42:40+00:00,2026-07-12T12:05:22+00:00,2026-07-11T13:51:38+00:00,2026-07-12T12:05:22+00:00
7,Google Maps,866,653,213,0,0,75.40%,24.60%,2026-07-08T21:16:36+00:00,2026-07-13T01:48:56+00:00,2026-07-09T21:23:33+00:00,2026-07-13T01:48:56+00:00
8,Netflix,742,576,166,0,0,77.63%,22.37%,2026-07-08T21:24:28+00:00,2026-07-13T01:26:29+00:00,2026-07-09T21:21:39+00:00,2026-07-13T01:26:29+00:00
9,Reddit,499,372,127,0,0,74.55%,25.45%,2026-07-08T21:18:41+00:00,2026-07-13T01:51:50+00:00,2026-07-09T21:17:07+00:00,2026-07-13T01:51:50+00:00



Timestamp validation checks:


,validation_check,passed
0,audit_rows_match_run_new_inserts,True
1,app_level_audit_counts_match_new_inserts,True
2,all_timestamp_classes_reconcile,True
3,all_10_apps_in_timestamp_summary,True
4,all_audited_rows_have_current_run_id,True


## 9. Export the Run B first-collection checkpoint

The updated database and all first-collection outputs are saved before the follow-up collection.

This checkpoint includes:

- the SQLite database after Run B first collection
- the app-level collection summary
- the detailed timestamp audit
- the app-level timestamp summary
- database and timestamp validation results
- the completed run record
- the pre-run snapshots
- prior run history and run metadata

The checkpoint will be used as the starting database for the Run B follow-up collection.

In [10]:
EXPORT_DIR = Path(
    "/content/runB_first_collection_checkpoint"
)

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

export_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_utc")

file_prefix = (
    "phase2_cadence_runB_first_collection"
)

# Save tabular outputs.
app_summary_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_app_summary.csv",
    index=False,
)

new_review_timestamp_audit_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_timestamp_audit.csv",
    index=False,
)

timestamp_summary_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_timestamp_summary.csv",
    index=False,
)

validation_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_database_validation.csv",
    index=False,
)

timestamp_validation_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_timestamp_validation.csv",
    index=False,
)

completed_run_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_completed_run_record.csv",
    index=False,
)

pre_run_snapshot_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_pre_run_snapshot.csv",
    index=False,
)

pre_run_app_snapshot_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_pre_run_app_snapshot.csv",
    index=False,
)

prior_runs_df.to_csv(
    EXPORT_DIR
    / f"{file_prefix}_prior_run_history.csv",
    index=False,
)

# Save an internally consistent SQLite backup.
checkpoint_db_path = (
    EXPORT_DIR
    / "google_play_reviews_after_runB_first_collection.sqlite"
)

if checkpoint_db_path.exists():
    checkpoint_db_path.unlink()

conn.commit()

with sqlite3.connect(
    checkpoint_db_path
) as backup_conn:
    conn.backup(backup_conn)

with sqlite3.connect(
    checkpoint_db_path
) as verify_conn:
    checkpoint_review_rows = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_raw
            """,
            verify_conn,
        )["n"].iloc[0]
    )

    checkpoint_run_count = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_ingestion_runs
            WHERE status = 'completed'
            """,
            verify_conn,
        )["n"].iloc[0]
    )

if checkpoint_review_rows != raw_rows_after:
    raise ValueError(
        "The checkpoint database review count "
        "does not match the active database."
    )

if checkpoint_run_count != 5:
    raise ValueError(
        f"Expected 5 completed runs in the checkpoint, "
        f"but found {checkpoint_run_count}."
    )


def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


database_sha256 = calculate_sha256(
    checkpoint_db_path
)

metadata = {
    "run_id": RUN_ID,
    "run_label": RUN_LABEL,
    "frequency_label": FREQUENCY_LABEL,
    "source": SOURCE,
    "language": LANGUAGE,
    "country": COUNTRY,
    "target_reviews_per_app": (
        TARGET_REVIEWS_PER_APP
    ),
    "app_count": len(app_config_df),
    "previous_run_id": PREVIOUS_RUN_ID,
    "previous_run_finished_at": (
        previous_run_finished_ts.isoformat()
    ),
    "run_started_at": (
        current_run_started_ts.isoformat()
    ),
    "run_finished_at": (
        current_run_finished_ts.isoformat()
    ),
    "gap_before_collection_hours": (
        gap_before_collection_hours
    ),
    "records_fetched_total": (
        records_fetched_total
    ),
    "new_records_inserted_total": (
        new_records_inserted_total
    ),
    "duplicates_skipped_total": (
        duplicates_skipped_total
    ),
    "processing_runtime_seconds": (
        processing_runtime_seconds
    ),
    "wall_clock_collection_seconds": (
        collection_runtime_seconds
    ),
    "review_rows_after": raw_rows_after,
    "database_size_after_mb": (
        db_size_after_mb
    ),
    "database_size_growth_mb": (
        db_size_growth_mb
    ),
    "posted_between_runs_total": (
        posted_between_runs_total
    ),
    "older_reviews_surfaced_later_total": (
        older_surfaced_total
    ),
    "timestamp_after_fetch_total": (
        timestamp_after_fetch_total
    ),
    "missing_or_unusable_timestamp_total": (
        missing_timestamp_total
    ),
    "database_sha256": database_sha256,
    "exported_at": datetime.now(
        timezone.utc
    ).isoformat(),
}

metadata_path = (
    EXPORT_DIR
    / f"{file_prefix}_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

# Compress the database inside the checkpoint.
database_zip_path = (
    EXPORT_DIR
    / "google_play_reviews_after_runB_first_collection.sqlite.zip"
)

with zipfile.ZipFile(
    database_zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    zf.write(
        checkpoint_db_path,
        arcname=checkpoint_db_path.name,
    )

checkpoint_db_path.unlink()

# Create a simple manifest after all checkpoint files exist.
manifest_rows = []

for file_path in sorted(EXPORT_DIR.iterdir()):
    if file_path.is_file():
        manifest_rows.append(
            {
                "file_name": file_path.name,
                "size_bytes": file_path.stat().st_size,
                "sha256": calculate_sha256(
                    file_path
                ),
            }
        )

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    EXPORT_DIR / "checkpoint_manifest.csv",
    index=False,
)

package_base_path = (
    BASE_DIR
    / (
        f"{file_prefix}_checkpoint_"
        f"{export_timestamp}"
    )
)

package_zip_path = Path(
    shutil.make_archive(
        str(package_base_path),
        "zip",
        root_dir=EXPORT_DIR,
    )
)

if not package_zip_path.exists():
    raise FileNotFoundError(
        "The checkpoint package was not created."
    )

print("Run B first-collection checkpoint created.")
print("-" * 72)
print("Checkpoint review rows:", f"{checkpoint_review_rows:,}")
print("Completed runs in database:", checkpoint_run_count)
print("Database SHA-256:", database_sha256)
print("Checkpoint package:", package_zip_path.name)
print(
    "Checkpoint package size:",
    f"{package_zip_path.stat().st_size / (1024 ** 2):.2f} MB",
)

print("\nCheckpoint files:")
display(
    pd.DataFrame(
        [
            {
                "file_name": path.name,
                "size_mb": (
                    path.stat().st_size
                    / (1024 ** 2)
                ),
            }
            for path in sorted(
                EXPORT_DIR.iterdir()
            )
            if path.is_file()
        ]
    )
)

files.download(str(package_zip_path))

Run B first-collection checkpoint created.
------------------------------------------------------------------------
Checkpoint review rows: 30,848
Completed runs in database: 5
Database SHA-256: 6616909712a83844e21c00010227133a3b1c4642c7414bbabac8555bc7820f17
Checkpoint package: phase2_cadence_runB_first_collection_checkpoint_20260714_020137_utc.zip
Checkpoint package size: 18.75 MB

Checkpoint files:


,file_name,size_mb
0,checkpoint_manifest.csv,0.001376
1,google_play_reviews_after_runB_first_collectio...,18.100438
2,phase2_cadence_runB_first_collection_app_summa...,0.002684
3,phase2_cadence_runB_first_collection_completed...,0.000581
4,phase2_cadence_runB_first_collection_database_...,0.000406
5,phase2_cadence_runB_first_collection_metadata....,0.001169
6,phase2_cadence_runB_first_collection_pre_run_a...,0.000903
7,phase2_cadence_runB_first_collection_pre_run_s...,0.000468
8,phase2_cadence_runB_first_collection_prior_run...,0.001333
9,phase2_cadence_runB_first_collection_timestamp...,2.831766


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>